# spheroid-seg — Colab GPU training

This notebook takes a fresh Colab runtime to a verified training result for
the v0.1 JAX/Flax U-Net. It works on both GPU and CPU runtimes: on GPU it
runs the full `configs/base.yaml` acceptance check, and on CPU it falls back
to a short, CPU-friendly smoke run.

**Before you start:** in Colab, go to *Runtime → Change runtime type* and
select a GPU (a T4 is enough for `configs/base.yaml`). Then run the cells
in order.

In [ ]:
# Setup constants and detect whether a GPU is present.
import os
import shutil
import subprocess
from pathlib import Path

REPO = Path(os.environ.get("SPHEROID_SEG_REPO", "/content/spheroid-seg"))
HAS_GPU = (
    shutil.which("nvidia-smi") is not None
    and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
)
UV_EXTRA = "cuda12" if HAS_GPU else ""
print(f"REPO={REPO}")
print(f"HAS_GPU={HAS_GPU}")
print(f"UV_EXTRA={UV_EXTRA!r}")

## Clone the repository

The notebook drives everything through subprocesses inside the cloned repo;
the Colab kernel itself stays stock. Every subprocess call below passes the
repo directory explicitly so cell ordering is the only ordering assumption.

In [ ]:
import subprocess

REPO.parent.mkdir(parents=True, exist_ok=True)
subprocess.run(
    ["git", "clone", "https://github.com/edgardomarchi/spheroid-seg.git", str(REPO)],
    check=True,
)

## Install uv

Bootstrap uv into the kernel environment so that plain `uv` is on PATH for
all later subprocess calls.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

## Sync the project

On GPU we add the CUDA 12 JAX extra; on CPU we sync the base project only.
The dev group is included so the smoke test can run.

In [ ]:
import subprocess

cmd = ["uv", "sync", "--group", "dev"]
if UV_EXTRA:
    cmd.extend(["--extra", UV_EXTRA])
subprocess.run(cmd, cwd=REPO, check=True)

## Verify JAX sees the device

Expected output on GPU: a list containing one `CudaDevice`. On CPU: a list
containing one `CpuDevice`.

In [ ]:
import subprocess

subprocess.run(
    ["uv", "run", "python", "-c", "import jax; print(jax.devices())"],
    cwd=REPO,
    check=True,
)

## Clean-room smoke test

Run the test suite. This doubles as the design-doc §5 clean-room
reproducibility check on a fresh machine.

In [ ]:
import subprocess

subprocess.run(["uv", "run", "pytest", "-q"], cwd=REPO, check=True)

## Pending acceptance check: overfit-one-batch

On a GPU the full `configs/base.yaml` check should run and the loss should
fall monotonically to near-zero, showing that the 512² / 7.7M-param model
can memorize a single batch. On CPU this cell is skipped because it is
impractical on CPU.

In [ ]:
import subprocess

if HAS_GPU:
    subprocess.run(
        [
            "uv",
            "run",
            "python",
            "-m",
            "spheroid_seg.train",
            "--config",
            "configs/base.yaml",
            "--overfit-one-batch",
        ],
        cwd=REPO,
        check=True,
    )
else:
    print("No GPU detected: skipping base.yaml --overfit-one-batch (impractical on CPU).")

## Short full-training sanity check

On GPU we run a few epochs of `configs/base.yaml` on the synthetic fallback
to measure real GPU speed. On CPU we run `configs/tiny.yaml` for a fast
smoke run. Replace this with a real `data/` upload (see the commented
Drive-mount cell below) for production training.

In [ ]:
import subprocess

if HAS_GPU:
    config, epochs = "configs/base.yaml", 3
else:
    config, epochs = "configs/tiny.yaml", 2
    print(f"No GPU detected: running CPU-friendly training ({config}, {epochs} epochs).")

subprocess.run(
    [
        "uv",
        "run",
        "python",
        "-m",
        "spheroid_seg.train",
        "--config",
        config,
        "--epochs",
        str(epochs),
    ],
    cwd=REPO,
    check=True,
)

## Download checkpoints before the session dies

Colab sessions can be cut at any time. Zip the latest run and download it.
Outside Colab the archive is created but the download step is skipped.

In [ ]:
import subprocess
from pathlib import Path

try:
    from google.colab import files

    has_colab = True
except ModuleNotFoundError:
    has_colab = False

prefix = "base_" if HAS_GPU else "tiny_"
run_dirs = sorted((REPO / "outputs" / "runs").glob(f"{prefix}*"))
if not run_dirs:
    raise RuntimeError("No training run found in outputs/runs/")
latest_run = run_dirs[-1]
archive = REPO.parent / "spheroid-seg-checkpoints.zip"
subprocess.run(["zip", "-r", str(archive), str(latest_run)], cwd=REPO, check=True)
if has_colab:
    files.download(str(archive))
else:
    print(f"Checkpoint archive created at {archive}")

## Real-data workflow (commented out)

For real training, mount Drive and copy your `data/` directory into the
repo. Un-comment and adapt the cell below.

In [ ]:
# from google.colab import drive
# import subprocess
# drive.mount("/content/drive")
# subprocess.run(
#     ["rsync", "-av", "/content/drive/MyDrive/spheroid-seg/data/",
#      str(REPO / "data")],
#     check=True,
# )
# subprocess.run(
#     ["uv", "run", "python", "-m", "spheroid_seg.train",
#      "--config", "configs/base.yaml"],
#     cwd=REPO,
#     check=True,
# )